
#### AUTOLOADER

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#from spotify_dab.utils.transformations import var_nithya

#if you directly use above way we get an error - ModuleNotFoundError: No module named 'spotify_dab'

#so you need to set the system path

import os
import sys

project_path = os.path.abspath(os.path.join(os.getcwd(),"..", ".."))
sys.path.append(project_path)

from utils.transformations import Reusable


###DimUser

In [0]:
checkpoint = "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimUser/checkpoint"

In [0]:
# Clear old checkpoint if desired
#dbutils.fs.rm(checkpoint, recurse=True)


In [0]:
#in autoloader we directly do not give format as parquet - we give it as cloud files initially and then in options we give parquet

df_user = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimUser/checkpoint")\
                .option("schemaEvolutionMode", "addNewColumns")\
                .load("abfss://bronze@storageazureprojectnit.dfs.core.windows.net/DimUser")

In [0]:
df_user_obj = Reusable()

df_user = df_user.withColumn("user_name", upper(col("user_name")))
df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])

df_user.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimUser/checkpoint")\
    .queryName("query_user")\
    .trigger(availableNow=True)\
    .option("path", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.dimUser")




In [0]:
display(spark.table("spotify_cata.silver.dimUser"))

###DimArtist

In [0]:
df_art = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimArtist/checkpoint")\
                .option("schemaEvolutionMode", "addNewColumns")\
                .load("abfss://bronze@storageazureprojectnit.dfs.core.windows.net/DimArtist")

In [0]:
df_artist_obj = Reusable()

df_art = df_artist_obj.dropColumns(df_art, ['_rescued_data'])
df_art = df_art.dropDuplicates(['artist_id'])

df_art.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimArtist/checkpoint" )\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.dimArtist")



In [0]:
display(spark.table("spotify_cata.silver.dimArtist"))

### DimTrack

In [0]:
df_track = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimTrack/checkpoint")\
                .option("schemaEvolutionMode", "addNewColumns")\
                .load("abfss://bronze@storageazureprojectnit.dfs.core.windows.net/DimTrack")

In [0]:
df_track_obj = Reusable()

df_track = df_track_obj.dropColumns(df_track, ['_rescued_data'])
df_track = df_track.withColumn("durationFlag", when(col('duration_sec')<150, 'low')\
                                                .when(col('duration_sec')<300, 'medium')\
                                                .otherwise('high'))
df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"), '-', ' '))


df_track.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimTrack/checkpoint" )\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.dimTrack")

In [0]:
display(spark.table("spotify_cata.silver.dimTrack"))

### DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimDate/checkpoint")\
                .option("schemaEvolutionMode", "addNewColumns")\
                .load("abfss://bronze@storageazureprojectnit.dfs.core.windows.net/DimDate")

In [0]:
df_date_obj = Reusable()

df_date = df_date_obj.dropColumns(df_date, ['_rescued_data'])


df_date.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimDate/checkpoint" )\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.dimDate")

In [0]:
display(spark.table("spotify_cata.silver.dimDate"))

### FactTable

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/FactStream/checkpoint")\
                .option("schemaEvolutionMode", "addNewColumns")\
                .load("abfss://bronze@storageazureprojectnit.dfs.core.windows.net/FactStream")

In [0]:
df_fact_obj = Reusable()

df_fact = df_fact_obj.dropColumns(df_fact, ['_rescued_data'])


df_fact.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation","abfss://silver@storageazureprojectnit.dfs.core.windows.net/FactStream/checkpoint" )\
    .trigger(availableNow=True)\
    .option("path", "abfss://silver@storageazureprojectnit.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.FactStream")

In [0]:
display(spark.table("spotify_cata.silver.FactStream"))

In [0]:
%sql
SELECT * FROM spotify_cata.gold.dimtrack
WHERE track_id IN (46,5)